# 🏫 AI와 함께 만든 학생·교사 평가 플랫폼 「참교육」 프로젝트 개발 기록

## 1. 프로젝트를 시작하며

이번 팀 프로젝트에서는 **학생과 교사의 평가 및 팀 편성을 관리하는 웹 플랫폼**을 기획하고 개발했다.

프로젝트 이름은 **「참교육」**으로 정했다.

처음에는 단순히 학생들을 관리하고 평가하는 서비스를 생각했지만, 기획을 구체화하면서 다음과 같은 문제를 해결하는 방향으로 발전했다.

> 학생들의 팀을 어떻게 편성할 것인가?  
> 학생끼리 평가할 때 자기 자신이나 자기 팀을 평가하는 문제는 어떻게 막을 것인가?  
> 같은 사람을 여러 번 평가하는 것은 어떻게 방지할 것인가?  
> 학생 평가와 선생님 평가를 어떻게 합산할 것인가?  
> 이전 평가 결과를 다음 팀 편성에 어떻게 활용할 것인가?

결국 참교육은 단순한 점수 입력 프로그램이 아니라,

**학생 관리 → 프로젝트 생성 → 팀 편성 → 평가 → 점수 계산 → 순위 산출 → 다음 팀 편성**

까지 연결되는 시스템을 목표로 만들게 되었다.

---

# 2. 프로젝트의 핵심 목표

참교육에서 가장 중요하게 생각한 것은 **평가 과정에서 발생할 수 있는 실수를 시스템이 최대한 막아주는 것**이었다.

예를 들어 학생이 직접 평가 대상을 선택하도록만 만들어 놓으면 다음과 같은 문제가 발생할 수 있다.

- 자기 자신을 평가할 수 있음
- 자신의 팀을 팀 평가할 수 있음
- 같은 대상을 여러 번 평가할 수 있음
- 다른 팀의 학생을 개인 평가할 수 있음
- 평가가 시작된 이후 문항이 변경될 수 있음
- 팀이 변경되면서 평가 대상이 섞일 수 있음

따라서 단순히 화면에서 버튼을 숨기는 것이 아니라 **DB와 서버에서도 이러한 상황을 방지하도록 설계하는 것**이 중요했다.

---

# 3. 사용자는 크게 학생과 관리자로 구분

서비스 사용자는 크게 두 종류로 나누었다.

```text
사용자
│
├── 학생
│
└── 관리자 / 교사
```

학생은 자신의 팀과 평가 정보를 확인하고 평가에 참여한다.

관리자는 평가 회차를 만들고, 학생을 팀으로 편성하고, 평가 진행 상황과 결과를 관리한다.

---

# 4. 학생 서비스 흐름

학생의 전체적인 서비스 흐름은 다음과 같이 설계했다.

```text
로그인
  ↓
학생 홈
  ↓
현재 평가 회차 확인
  ↓
소속 팀 확인
  ↓
팀원 확인
  ↓
팀 평가
  ↓
개인 평가
  ↓
평가 완료 여부 확인
  ↓
평가 종료
  ↓
공개된 점수 / 석차 / 팀 결과 확인
```

학생 홈에서는 현재 상황을 한눈에 확인할 수 있도록 했다.

예를 들어 현재 팀이 있는 학생이라면:

```text
현재 평가 회차
소속 팀
팀원
평가 진행률
현재 1위 팀
```

등을 보여준다.

아직 프로젝트나 팀이 배정되지 않은 학생이라면:

```text
현재 진행 중인 프로젝트가 없습니다.

현재 소속 팀 없음
지난 평가 결과
```

등을 보여주는 방향으로 설계했다.

---

# 5. 관리자 서비스 흐름

관리자는 학생보다 훨씬 많은 기능을 사용한다.

전체 흐름은 다음과 같이 설계했다.

```text
관리자 대시보드
        ↓
평가 회차 생성
        ↓
평가 문항 설정
        ↓
팀 편성
        ↓
관리자 검토 / 수정
        ↓
평가 시작
        ↓
제출 현황 확인
        ↓
평가 마감
        ↓
점수 계산
        ↓
석차 계산
        ↓
결과 검토
        ↓
결과 공개
        ↓
다음 평가 팀 편성
```

평가 회차를 만들 때는 다음과 같은 정보를 관리한다.

```text
평가명
과제명
시작일
종료일
팀 수
평가 상태
```

---

# 6. 평가 회차 상태 관리

평가에는 상태가 필요하다고 판단했다.

대표적으로 다음과 같이 나누었다.

```text
READY
IN_PROGRESS
ENDED
```

### READY

아직 평가가 시작되지 않은 상태다.

이 단계에서는 관리자가 평가 문항이나 팀을 수정할 수 있다.

### IN_PROGRESS

학생들이 실제로 평가하고 있는 상태다.

이때부터는 평가 문항을 함부로 수정하면 안 된다.

예를 들어 처음 평가 문항이:

```text
협업을 잘했는가?
```

였는데 평가 도중:

```text
발표를 잘했는가?
```

로 변경되면 같은 점수라도 의미가 완전히 달라진다.

따라서 **평가가 시작된 이후에는 기존 평가의 의미를 보호하기 위해 문항 변경을 제한**하도록 설계했다.

### ENDED

평가가 종료된 상태다.

이후 점수 계산과 결과 공개가 진행된다.

---

# 7. 평가 종류

참교육에서는 평가를 크게 다음과 같이 구분했다.

```text
학생 팀 평가
학생 개인 평가
교사 평가
```

학생 평가만으로 최종 점수를 결정하면 인기투표처럼 변할 가능성이 있기 때문에 교사의 평가도 함께 반영하도록 했다.

최종적으로 설정한 가중치는:

```text
학생 팀 평가    30%
학생 개인 평가  30%
교사 평가       40%
```

이다.

즉 개념적으로 최종 점수는 다음과 같다.

```text
최종 점수

= 학생 팀 평가 × 0.3
+ 학생 개인 평가 × 0.3
+ 교사 평가 × 0.4
```

---

# 8. 팀 평가의 핵심 규칙

팀 평가는 학생이 다른 팀을 평가하는 기능이다.

가장 중요한 규칙은:

> **자신의 팀은 평가할 수 없다.**

예를 들어 내가 1팀이라면:

```text
1팀 → 평가 불가능
2팀 → 평가 가능
3팀 → 평가 가능
4팀 → 평가 가능
```

이렇게 되어야 한다.

따라서 화면에서 평가할 팀을 보여줄 때부터 자신의 팀을 제외하도록 설계했다.

하지만 화면에서만 막는 것으로는 충분하지 않다.

사용자가 직접 요청을 조작할 수도 있기 때문에 서버에서도:

```text
평가자의 팀 ID
        ==
평가 대상 팀 ID
```

인지 확인하고 같다면 평가를 거부하는 방식이 필요하다.

---

# 9. 개인 평가의 핵심 규칙

개인 평가는 같은 팀에 속한 팀원끼리 진행한다.

여기에서도 중요한 규칙이 있다.

> **자기 자신은 평가할 수 없다.**

예를 들어 1팀이:

```text
민수
철수
영희
지수
```

이고 내가 `민수`라면 평가 대상은:

```text
철수
영희
지수
```

만 보여야 한다.

따라서 개인 평가 대상을 조회할 때는:

```text
같은 팀인가?
        ↓
YES

본인인가?
        ↓
YES → 제외
NO  → 평가 가능
```

이라는 논리가 필요하다.

---

# 10. 중복 평가 방지

평가 시스템에서 특히 중요하게 생각했던 부분이다.

학생이 같은 사람이나 같은 팀을 여러 번 평가하면 결과가 왜곡될 수 있다.

예를 들어:

```text
민수 → 철수 평가 완료

민수 → 철수 다시 평가
```

가 가능하면 안 된다.

따라서 DB에서 다음과 같은 조합을 기준으로 중복을 확인할 수 있다.

```text
평가 회차
평가자
평가 대상
평가 문항
```

이미 동일한 평가가 존재한다면 새로운 평가가 저장되지 않도록 해야 한다.

즉 중복 방지는:

```text
화면
+
서버 로직
+
DB 제약조건
```

여러 단계에서 처리하는 것이 안전하다.

---

# 11. DB 담당으로 프로젝트 참여

이번 프로젝트에서 내가 맡은 주요 역할은 **DB 설계와 데이터 구조 관리**였다.

처음에는 단순히 테이블을 만드는 것이라고 생각했지만 실제로 진행해보니 DB 설계는 서비스 전체 흐름과 밀접하게 연결되어 있었다.

예를 들어:

> 학생이 어느 팀인가?

라는 질문 하나만 해도:

```text
학생
↓
팀 소속 정보
↓
팀
↓
평가 회차
↓
프로젝트
```

처럼 여러 데이터의 관계를 생각해야 했다.

---

# 12. 주요 DB 테이블

프로젝트를 진행하면서 다음과 같은 데이터 구조를 사용했다.

```text
students
teachers
projects
teams
team_members
evaluation_rounds
evaluation_questions
team_evaluation_scores
individual_evaluation_scores
teacher_team_scores
teacher_individual_scores
```

각 테이블은 서로 다른 역할을 담당한다.

---

## students

학생 정보를 관리한다.

예를 들면:

```text
학생 ID
이름
이메일
전화번호
Slack ID
비밀번호
```

등의 정보를 관리할 수 있다.

---

## teachers

교사 정보를 관리한다.

학생과 교사는 서비스에서 사용할 수 있는 기능과 권한이 다르기 때문에 별도로 관리하는 방향을 사용했다.

---

## projects

프로젝트 정보를 관리한다.

예를 들어:

```text
참교육 프로젝트
쇼핑몰 프로젝트
데이터 분석 프로젝트
```

처럼 여러 프로젝트가 존재할 수 있다.

---

## evaluation_rounds

평가 회차를 관리한다.

예를 들어:

```text
1차 평가
2차 평가
3차 평가
```

와 같은 개념이다.

평가 회차에는:

```text
READY
IN_PROGRESS
ENDED
```

와 같은 상태도 저장된다.

---

## teams

각 평가 회차의 팀 정보를 저장한다.

예를 들어:

```text
1팀
2팀
3팀
4팀
```

등이다.

---

## team_members

학생과 팀의 관계를 저장한다.

이 테이블이 중요한 이유는 학생 정보 자체에 단순히:

```text
team = 1
```

처럼 저장해버리면 과거 팀 정보를 관리하기 어려워질 수 있기 때문이다.

별도의 관계 테이블을 사용하면:

```text
학생 A

1차 평가 → 1팀
2차 평가 → 3팀
3차 평가 → 2팀
```

처럼 과거 팀 기록도 관리할 수 있다.

---

# 13. 평가 문항 관리

평가 문항은 별도의 테이블에서 관리하도록 설계했다.

예를 들어:

```text
협업을 잘했는가?
발표 준비가 충분했는가?
역할을 성실하게 수행했는가?
의사소통을 잘했는가?
팀에 기여했는가?
```

등이다.

문항에는 평가 종류를 구분하는 값도 필요하다.

```text
TEAM
INDIVIDUAL
```

이를 통해 같은 평가 시스템에서도 팀 평가 문항과 개인 평가 문항을 구분할 수 있다.

---

# 14. 평가 점수 테이블

학생 팀 평가 점수와 개인 평가 점수는 성격이 다르기 때문에 별도로 관리했다.

```text
team_evaluation_scores
individual_evaluation_scores
```

교사 평가 역시 별도의 구조로 관리했다.

```text
teacher_team_scores
teacher_individual_scores
```

이렇게 분리하면 나중에:

```text
학생 팀 평가 평균
학생 개인 평가 평균
교사 평가 평균
```

을 각각 계산한 뒤 가중치를 적용할 수 있다.

---

# 15. ERD를 공부하면서 알게 된 점

이번 프로젝트를 통해 ERD가 단순한 DB 그림이 아니라는 것을 알게 되었다.

ERD는 쉽게 말하면:

> **데이터끼리 어떤 관계를 가지고 있는지를 그림으로 표현한 것**

이다.

예를 들어:

```text
STUDENT
   │
   │ 소속
   ↓
TEAM_MEMBER
   │
   ↓
TEAM
```

이라는 관계를 생각할 수 있다.

그리고:

```text
TEAM
   │
   ↓
EVALUATION_ROUND
   │
   ↓
PROJECT
```

처럼 관계가 계속 이어질 수 있다.

DB를 설계하면서 가장 중요했던 것은 **현재 화면에 필요한 데이터만 생각하지 않는 것**이었다.

현재 팀만 저장하면 당장은 편하지만, 다음 평가가 시작되면 이전 팀 정보를 잃어버릴 수 있다.

그래서:

> **"이 데이터를 나중에도 기록으로 남겨야 하는가?"**

를 계속 생각해야 했다.

---

# 16. PostgreSQL 사용

DB는 PostgreSQL을 사용했다.

프로젝트를 진행하면서 스키마도 직접 생성하고 변경했다.

초기에는:

```text
practice
```

스키마에서 작업하다가 이후 프로젝트 구조에 맞춰:

```text
cham_edu
```

스키마를 사용하는 방향으로 변경했다.

이 과정에서 여러 SQL 오류도 경험했다.

---

# 17. 개발하면서 만난 DB 오류

실제 프로젝트를 진행하면서 SQL이 생각보다 작은 문법 차이에도 민감하다는 것을 알게 되었다.

대표적으로 경험했던 오류는:

```text
relation does not exist
relation already exists
INSERT target columns보다 expressions가 많음
UNIQUE 관련 오류
BOOLEAN 관련 오류
created_at 관련 오류
```

등이었다.

처음에는 오류 메시지가 굉장히 어렵게 느껴졌지만 하나씩 확인하면서 결국 오류 메시지도 **문제가 발생한 위치를 알려주는 힌트**라는 것을 알게 되었다.

예를 들어:

```text
relation does not exist
```

가 발생한다면 단순히 "DB가 고장났다"라고 생각하는 것이 아니라:

```text
테이블이 실제로 존재하는가?
현재 schema가 맞는가?
search_path가 맞는가?
테이블 이름이 정확한가?
```

를 확인해야 했다.

---

# 18. Django와 기존 DB 연결

웹 개발에는 Python과 Django를 사용했다.

초기에는 Django가 자체적으로 DB 테이블을 만드는 방식도 사용했지만, 기존 PostgreSQL 테이블과 연결해야 하는 경우도 있었다.

이 과정에서 Django 모델과 실제 DB 테이블의 관계를 공부했다.

기존 DB 테이블을 Django에서 사용하는 경우:

```python
class Meta:
    managed = False
```

와 같은 설정도 사용할 수 있다는 것을 알게 되었다.

즉:

> "이 테이블은 Django가 새로 만들고 관리하는 것이 아니라 이미 존재하는 테이블이다."

라는 의미로 사용할 수 있다.

---

# 19. CRUD 구현

학생 또는 회원 데이터를 관리하기 위해 CRUD 기능도 구현했다.

CRUD는:

```text
Create
Read
Update
Delete
```

의 약자다.

한국어로 표현하면:

```text
등록
조회
수정
삭제
```

이다.

실습에서는 다음과 같은 URL 구조를 사용했다.

```text
/members/

/members/create/

/members/<id>/update/

/members/<id>/delete/
```

이를 통해 웹에서 실제 DB 데이터를:

```text
조회하고
↓
추가하고
↓
수정하고
↓
삭제하는
```

과정을 경험했다.

---

# 20. 삭제 기능에서 생각했던 부분

삭제 기능을 구현하면서 단순히 삭제 버튼을 누르는 즉시 데이터를 지우는 것은 위험하다고 생각했다.

그래서:

```text
삭제 버튼 클릭
        ↓
"정말 삭제하시겠습니까?"
        ↓
확인
        ↓
삭제
```

방식을 사용했다.

작은 기능이지만 실제 서비스에서는 사용자의 실수를 막는 것도 중요한 기능이라는 것을 알게 되었다.

---

# 21. 로그인과 세션

학생 로그인 기능도 구현했다.

로그인이 성공하면 세션에 다음과 같은 정보를 저장하는 방식을 사용했다.

```text
user_type
student_id
user_name
```

로그인 이후에는 이 정보를 이용해:

```text
현재 로그인한 학생은 누구인가?
현재 학생의 팀은 어디인가?
어떤 평가를 할 수 있는가?
```

등을 판단할 수 있다.

즉 로그인은 단순히 아이디와 비밀번호가 맞는지 확인하는 것으로 끝나는 것이 아니었다.

로그인 이후 **현재 사용자가 누구인지 계속 기억하는 것**도 필요했다.

---

# 22. 회원가입

회원가입에서는 다음과 같은 내용을 확인했다.

```text
비밀번호 확인
학생 계정 중복 확인
교사 계정 중복 확인
회원 정보 저장
회원가입 완료 후 로그인 페이지 이동
```

회원가입 역시 단순히 INSERT를 하는 것이 아니라 **잘못된 데이터가 들어가지 않도록 검증하는 과정**이 중요했다.

---

# 23. 화면 구성

학생 화면은 다음과 같이 구성했다.

```text
로그인
아이디 / 비밀번호 찾기
회원가입
학생 홈
내 팀
팀 평가
개인 평가
리포트
설정 / 마이페이지
```

관리자 화면에서는:

```text
관리자 대시보드
평가 회차 관리
평가 문항 관리
팀 편성
평가 진행 현황
점수 및 순위
결과 관리
```

등의 기능이 필요했다.

---

# 24. Bootstrap을 이용한 UI

프로젝트 UI에서는 별도의 CSS를 복잡하게 작성하기보다는 **Bootstrap 클래스를 활용하는 방향**으로 작업했다.

이를 통해:

```text
버튼
카드
테이블
모달
폼
레이아웃
```

등을 빠르게 구성할 수 있었다.

특히 프로젝트 기간이 짧은 상황에서는 디자인을 처음부터 모두 만드는 것보다 Bootstrap을 이용해 **기능 구현과 화면 가독성에 집중**할 수 있었다.

---

# 25. 팀 자동 편성

참교육 프로젝트에서 중요한 기능 중 하나는 **다음 평가를 위한 팀 편성**이다.

단순한 랜덤 팀 편성도 가능하지만 프로젝트가 진행될수록 다음과 같은 문제가 생긴다.

```text
이전에 같은 팀이었던 학생들이
또 같은 팀이 될 수 있음
```

따라서 다음 팀 편성에서는 과거 팀 기록을 확인할 필요가 있다.

개념적으로는:

```text
이전 팀 기록 조회
        ↓
같은 팀 경험 확인
        ↓
중복 조합 최소화
        ↓
학생들의 평가 결과 확인
        ↓
팀별 균형 고려
        ↓
새로운 팀 생성
```

과 같은 구조를 생각할 수 있다.

이 기능을 제대로 구현하려면 단순한 랜덤보다 훨씬 많은 데이터가 필요하다는 것을 알게 되었다.

---

# 26. 평가 도중 팀이 섞이는 문제

실제 데이터를 테스트하면서 **평가 도중 팀 정보가 예상과 다르게 연결되는 상황**도 확인했다.

이 문제를 확인하기 위해 DB에서:

```text
학생
팀
평가 회차
평가 데이터
```

를 함께 조회해 실제로 어느 학생이 어느 팀에 속해 있었는지 확인했다.

이 과정에서 DB를 직접 조회할 수 있다는 것이 디버깅에 굉장히 중요하다는 것을 알게 되었다.

화면만 보면:

```text
"뭔가 이상하다."
```

정도로 끝날 수 있지만 SQL로 데이터를 확인하면:

```text
어느 학생인가?
어느 회차인가?
어느 팀인가?
어떤 평가 데이터가 연결되었는가?
```

를 구체적으로 추적할 수 있다.

---

# 27. SQL이 개발에서 왜 중요한지 알게 됨

처음에는 Django를 사용하면 SQL을 직접 사용할 일이 별로 없을 것이라고 생각했다.

하지만 실제 프로젝트에서는:

```text
특정 학생 조회
특정 학생의 평가 점수 확인
현재 팀 조회
과거 팀 조회
평가 데이터 확인
중복 데이터 확인
잘못 연결된 데이터 확인
```

등을 위해 SQL을 직접 사용하는 일이 많았다.

ORM이 편리하더라도 **DB 안에서 실제 데이터가 어떻게 저장되어 있는지 이해하는 능력**이 중요하다는 것을 알게 되었다.

---

# 28. 프로젝트를 하면서 가장 크게 배운 것

처음에는 웹 서비스를 이렇게 생각했다.

```text
화면
+
Python
+
DB
```

하지만 실제로 만들어보니 이 세 가지가 독립적으로 존재하는 것이 아니었다.

예를 들어 학생이 평가 버튼을 누르면:

```text
학생이 평가 버튼 클릭
        ↓
HTML Form
        ↓
Django View
        ↓
조건 검사
        ↓
DB 조회
        ↓
평가 가능 여부 판단
        ↓
DB 저장
        ↓
결과 반환
        ↓
화면 변경
```

이라는 여러 과정이 연결된다.

즉 사용자는 버튼 하나를 누르지만 그 뒤에서는 여러 기능이 동시에 움직이고 있었다.

---

# 29. DB 설계가 서비스 규칙을 만든다

이번 프로젝트에서 특히 인상적이었던 부분이다.

DB는 단순히 데이터를 저장하는 장소라고 생각했는데 실제로는 서비스의 규칙과 매우 밀접했다.

예를 들어:

```text
자기 자신 평가 금지
본인 팀 평가 금지
중복 평가 금지
평가 시작 후 문항 수정 금지
과거 팀 기록 유지
```

같은 규칙을 제대로 구현하려면 DB 구조가 이를 지원해야 한다.

즉:

> **좋은 DB 설계는 데이터를 잘 저장하는 것뿐 아니라 잘못된 데이터가 들어가는 것을 막는 것까지 고려해야 한다.**

는 것을 배웠다.

---

# 30. 기획 → DB → Backend → 화면의 연결

이번 프로젝트를 진행하면서 개발 순서도 조금 이해하게 되었다.

```text
서비스에서 무엇을 할 것인가?
        ↓
기획

어떤 정보가 필요한가?
        ↓
데이터 정의

데이터가 서로 어떤 관계인가?
        ↓
ERD

어떻게 저장할 것인가?
        ↓
DB

어떤 조건으로 처리할 것인가?
        ↓
Python / Django

사용자에게 어떻게 보여줄 것인가?
        ↓
HTML / Bootstrap
```

처음에는 각각 별개의 공부라고 생각했지만 실제 프로젝트를 해보니 전부 연결되어 있었다.

---

# 31. 프로젝트를 통해 조건문의 중요성도 다시 알게 됨

파이썬에서 배우는 간단한 `if`문이 실제 웹서비스에서도 그대로 사용된다는 점도 흥미로웠다.

초보 단계에서는:

```python
if score >= 80:
    print("합격")
```

같은 문제를 풀지만 실제 서비스에서는 개념적으로:

```text
IF 현재 사용자가 본인인가?
    평가 금지

IF 이미 평가했는가?
    중복 평가 금지

IF 평가 상태가 READY인가?
    문항 수정 가능

IF 평가 상태가 IN_PROGRESS인가?
    문항 수정 제한
```

처럼 사용된다.

결국 지금 배우고 있는 조건문이 실제 서비스의 **규칙을 코드로 표현하는 기본 도구**라는 것을 알게 되었다.

---

# 32. 리스트와 반복문 역시 실제 프로젝트와 연결된다

리스트와 `for`문 역시 단순한 연습 문제가 아니었다.

예를 들어 학생 목록이 있다면:

```text
학생1
학생2
학생3
학생4
학생5
```

각 학생을 하나씩 확인하면서:

```text
현재 팀은 어디인가?
평가를 완료했는가?
점수는 얼마인가?
이전에 누구와 같은 팀이었는가?
```

등을 처리해야 한다.

결국:

```text
for
→ 여러 데이터를 하나씩 처리

if
→ 각각의 상황을 판단

DB
→ 필요한 데이터를 저장하고 조회
```

가 서로 연결된다.

---

# 33. 아직 개선할 부분

현재 프로젝트가 완성형이라고 생각하지는 않는다.

앞으로 개선한다면 다음과 같은 부분을 더 고민할 수 있을 것 같다.

```text
팀 자동 편성 알고리즘 개선

과거 팀 중복 최소화

평가 점수 편향 방지

평가 데이터 검증 강화

관리자 대시보드 개선

평가 결과 시각화

권한 관리 강화

예외 처리 강화

서비스 외부 배포

보안 강화
```

특히 로컬 환경에서는 잘 작동하더라도 외부 사용자가 실제로 접속하는 서비스가 되면:

```text
서버
도메인
HTTPS
DB 보안
환경변수
사용자 인증
접근 권한
백업
```

등 새로운 문제를 고려해야 한다.

---

# 34. 프로젝트를 마치며

이번 참교육 프로젝트를 진행하면서 가장 크게 느낀 점은 **개발은 단순히 코드를 작성하는 작업이 아니라는 것**이다.

처음에는:

> "어떤 코드를 작성해야 하지?"

라는 생각부터 했다.

하지만 프로젝트를 진행하면서 점점:

> "어떤 문제가 있는가?"

> "그 문제를 막으려면 어떤 규칙이 필요한가?"

> "그 규칙을 구현하려면 어떤 데이터가 필요한가?"

> "그 데이터를 어떤 관계로 저장해야 하는가?"

> "사용자가 잘못된 행동을 했을 때 시스템은 어떻게 막아야 하는가?"

를 먼저 생각하게 되었다.

특히 DB를 담당하면서 **화면 뒤에서 데이터가 어떻게 연결되고 움직이는지**를 직접 경험할 수 있었다.

---

# 📌 참교육 프로젝트 핵심 정리

| 구분 | 내용 |
|---|---|
| 프로젝트명 | 참교육 |
| 목적 | 학생·교사 평가 및 팀 편성 관리 |
| Backend | Python / Django |
| Database | PostgreSQL |
| Frontend | Django Template / Bootstrap |
| 주요 사용자 | 학생 / 교사·관리자 |
| 주요 기능 | 로그인, 팀 관리, 팀 평가, 개인 평가, 교사 평가, 점수 계산, 순위, 팀 편성 |
| 평가 상태 | READY / IN_PROGRESS / ENDED |
| 학생 팀 평가 | 30% |
| 학생 개인 평가 | 30% |
| 교사 평가 | 40% |
| 핵심 정책 | 자기평가 금지 / 본인 팀 평가 금지 / 중복 평가 방지 |
| DB 핵심 | 학생·팀·평가 회차·평가 기록 관계 관리 |
| UI | Bootstrap 중심 |
| 주요 학습 | ERD, SQL, PostgreSQL, Django, CRUD, 세션, DB 관계 |

---

# 💡 한 문장으로 정리

> **참교육 프로젝트는 학생과 교사의 평가 데이터를 관리하고, 공정한 평가와 다음 팀 편성을 지원하기 위해 기획한 학생·교사 평가 관리 플랫폼이다.**

이번 프로젝트를 통해 단순히 Python이나 SQL 문법을 배우는 것을 넘어 **기획한 서비스가 DB와 Backend, 화면을 거쳐 실제 기능으로 연결되는 과정**을 경험할 수 있었다.

#Python #Django #PostgreSQL #SQL #파이썬 #웹개발 #팀프로젝트 #DB설계 #ERD #Bootstrap #CRUD #개발공부 #참교육 #AI공부